# LC 39 — Combination Sum
**Day 53 | Theme: Backtracking + Tries | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">

**Core Insight:** Sort candidates, then backtrack from each index
allowing reuse of the same element. Prune immediately when a
candidate exceeds the remaining target — no need to explore
further siblings once the sorted list overshoots.

</div>

## Official Problem Statement

Given an array of **distinct** integers `candidates` and a target
integer `target`, return a list of all **unique combinations** of
`candidates` where the chosen numbers sum to `target`. You may
return the combinations in **any order**.

The **same** number may be chosen from `candidates` an **unlimited
number of times**. Two combinations are unique if the frequency of
at least one of the chosen numbers is different.

The test cases are generated such that the number of unique
combinations that sum up to `target` is less than `150`
combinations for the given input.

**Constraints:**
- `1 <= candidates.length <= 30`
- `2 <= candidates[i] <= 40`
- All elements of `candidates` are **distinct**.
- `1 <= target <= 40`

## What This Is Actually Asking

You have a set of unique numbers and a target. Find every
multiset of those numbers that adds up to exactly the target.
Unlike permutations, order does not matter — `[2,2,3]` and
`[3,2,2]` are the same combination. Each number can appear more
than once in a single combination, so you must allow re-picking.
Duplicates across combinations must be avoided, which is handled
by only moving forward in the sorted list, never backward.

## Walk Through an Example by Hand

```
candidates = [2, 3, 6, 7], target = 7
After sort: [2, 3, 6, 7]

backtrack(start=0, path=[], remain=7)
  try 2 → backtrack(start=0, path=[2], remain=5)
    try 2 → backtrack(start=0, path=[2,2], remain=3)
      try 2 → backtrack(start=0, path=[2,2,2], remain=1)
        try 2 → remain=-1, 2>1 → PRUNE (break)
      try 3 → backtrack(start=1, path=[2,2,3], remain=0)
        remain==0 → ADD [2,2,3] ✓
      try 6 → 6>3 → PRUNE
    try 3 → backtrack(start=1, path=[2,3], remain=2)
      try 3 → 3>2 → PRUNE
    try 6 → 6>5 → PRUNE
  try 3 → backtrack(start=1, path=[3], remain=4)
    try 3 → backtrack(start=1, path=[3,3], remain=1)
      try 3 → 3>1 → PRUNE
    try 6 → 6>4 → PRUNE
  try 6 → backtrack(start=2, path=[6], remain=1)
    try 6 → 6>1 → PRUNE
  try 7 → backtrack(start=3, path=[7], remain=0)
    remain==0 → ADD [7] ✓

Result: [[2,2,3], [7]]
```

## The Picture

```
candidates=[2,3,6,7]  target=7

                     root(remain=7)
          /           |         |       \
       pick 2       pick 3   pick 6   pick 7
      (rem=5)      (rem=4)  (rem=1)  (rem=0)*
      /    \         |
  pick2   pick3   pick3
 (rem=3)  (rem=2) (rem=1)
   /  \
pick2 pick3
(rem=1)(rem=0)*
  |
 PRUNE
(2>1)

* = solution found

KEY RULES:
  [1] remain == 0  → record path as answer
  [2] cand > remain → break (sorted, all rest bigger)
  [3] recurse with same start index → allows reuse
  [4] pop after recurse → restore path (backtrack)
```

## When To Use This Pattern

- When you need **all subsets / combinations** that satisfy a
  constraint, think **backtracking with a loop**.
- When elements can be **reused**, think **recurse with same
  index** instead of `start+1`.
- When the input is **sorted** and the target is numerical,
  think **prune with `break`** once a candidate exceeds remain.
- When order does **not** matter, think **forward-only index**
  to avoid duplicate combinations.
- When the solution space is **exponential** but prunable, think
  **DFS + backtrack** over BFS or DP.

## The Approach

Sort the candidates so that early pruning is possible. Use a
recursive helper that tracks a start index, the current path,
and remaining target. At each level, iterate from start onward;
if the current candidate exceeds remain, break immediately since
all further candidates are larger. If remain hits zero, snapshot
the path into results. Otherwise, append the candidate, recurse
with the same index (reuse allowed), then pop to backtrack.

In [2]:
from typing import List

In [3]:
def test_harness(func):
    """Run test cases for combination_sum."""
    def norm(combos):
        return sorted(tuple(sorted(c)) for c in combos)

    cases = [
        # (candidates, target, expected)
        ([2, 3, 6, 7],  7, [[2, 2, 3], [7]]),
        ([2, 3, 5],     8, [[2, 2, 2, 2], [2, 3, 3], [3, 5]]),
        ([2],           1, []),
        ([1],           1, [[1]]),
        ([1],           2, [[1, 1]]),
    ]

    passed = 0
    for i, (cands, target, expected) in enumerate(cases):
        result = func(cands, target)
        ok = norm(result) == norm(expected)
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"  Test {i+1}: {status} | "
            f"cands={cands} target={target}"
        )
        if not ok:
            print(f"    Expected: {sorted(expected)}")
            print(f"    Got:      {sorted(result)}")

    total = len(cases)
    print(f"\n  {passed}/{total} tests passed.")

In [5]:
def combination_sum(
    candidates: List[int], target: int
) -> List[List[int]]:
    result = []
    def backtrack(start, current,remaining):
        if remaining == 0:
            result.append(current[:])
            return
        if remaining < 0:
            return
        for i in range(start , len(candidates)):
            current.append(candidates[i])
            backtrack(i, current, remaining - candidates[i])
            current.pop()
    backtrack (0, [], target)
    return result
#Quick debug — run this cell while building
print(combination_sum([2,3,6,7], 7))   # [[2,2,3],[7]]
print(combination_sum([2,3,5], 8))     # [[2,2,2,2],[2,3,3],[3,5]]
print(combination_sum([2], 1))          # []
print(combination_sum([1], 2))          # [[1,1]]
test_harness(combination_sum)    

[[2, 2, 3], [7]]
[[2, 2, 2, 2], [2, 3, 3], [3, 5]]
[]
[[1, 1]]
  Test 1: PASSED | cands=[2, 3, 6, 7] target=7
  Test 2: PASSED | cands=[2, 3, 5] target=8
  Test 3: PASSED | cands=[2] target=1
  Test 4: PASSED | cands=[1] target=1
  Test 5: PASSED | cands=[1] target=2

  5/5 tests passed.


In [ ]:
# Uncomment and run when solution is ready
# test_harness(combination_sum)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force (all subsets with replacement) | O(n^(T/min)) | O(T/min) stack |
| Sorted backtrack + prune (optimal) | O(n^(T/min)) pruned | O(T/min) stack |

- `n` = number of candidates, `T` = target, `min` = smallest
  candidate value.
- Sorting does not change worst-case but prunes heavily in
  practice, cutting branches that can never reach the target.
- Result storage is O(k * S) where k = avg combo length, S =
  number of valid combinations (at most 150 per constraints).

## Real World Connection

At **Citi**, portfolio construction often requires finding all
subsets of financial instruments whose notional values sum to a
specific hedge target — a direct analogue of combination sum.
On **AWS**, cost-optimization engines enumerate combinations of
reserved instance types whose total capacity matches a workload
requirement. In **data engineering**, ETL dependency resolution
can require selecting repeated pipeline stages that together
satisfy a throughput constraint. Backtracking with pruning keeps
these searches tractable by cutting entire subtrees early,
mimicking how a human analyst would skip obviously over-budget
options without evaluating them further.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra